Found stat page: https://www.ncaa.com/stats/football/fbs/current/individual/7
  Rank            Name              Team   Cl Position   G  Rush  Rush Yds  \
0    1        Cam Cook  Jacksonville St.  Jr.       RB  13   295      1659   
1    2     Ahmad Hardy          Missouri  So.       RB  13   256      1649   
2    3    Evan Dickens           Liberty  So.       RB  11   229      1339   
3    4  Emmett Johnson          Nebraska  Jr.       RB  12   251      1451   
4    5  Jeremiyah Love        Notre Dame  Jr.       RB  12   199      1372   

   Rush TD    YPG  
0       16  127.6  
1       16  126.8  
2       16  121.7  
3       12  120.9  
4       18  114.3  
Found stat page: https://www.ncaa.com/stats/football/fbs/current/individual/8
   Rank              Name         Team   Cl Position   G  Pass Att  Pass Com  \
0     1  Fernando Mendoza      Indiana  Jr.       QB  15       352       257   
1     2      Julian Sayin     Ohio St.  So.       QB  14       391       301   
2     3       D

In [4]:
import pandas as pd
import requests
import time

# webscraping for all divisions
divisions = {
    "FBS": "https://www.ncaa.com/stats/football/fbs/current/individual/{}",
    "FCS": "https://www.ncaa.com/stats/football/fcs/current/individual/{}",
    "D2":  "https://www.ncaa.com/stats/football/d2/current/individual/{}",
    "D3":  "https://www.ncaa.com/stats/football/d3/current/individual/{}"
}

stat_ids = range(1, 150)

# list to store all tables
all_tables = []

for division, base_url in divisions.items():

    for stat_id in stat_ids:
        url = base_url.format(stat_id)

        try:
            tables = pd.read_html(url)

            if tables:
                df = tables[0]

                df["division"] = division
                df["stat_id"] = stat_id
                df["source_url"] = url

                print("Found stat page:", url)
                print(df.head())

                all_tables.append(df)

        except:
            pass

        time.sleep(0.05)

df_all = pd.concat(all_tables, ignore_index=True, sort=False)

print("\nFINAL SUMMARY")
print("Total statistics scraped:", len(all_tables))
print("Total rows scraped:", len(df_all))

Found stat page: https://www.ncaa.com/stats/football/fbs/current/individual/7
  Rank            Name              Team   Cl Position   G  Rush  Rush Yds  \
0    1        Cam Cook  Jacksonville St.  Jr.       RB  13   295      1659   
1    2     Ahmad Hardy          Missouri  So.       RB  13   256      1649   
2    3    Evan Dickens           Liberty  So.       RB  11   229      1339   
3    4  Emmett Johnson          Nebraska  Jr.       RB  12   251      1451   
4    5  Jeremiyah Love        Notre Dame  Jr.       RB  12   199      1372   

   Rush TD    YPG division  stat_id  \
0       16  127.6      FBS        7   
1       16  126.8      FBS        7   
2       16  121.7      FBS        7   
3       12  120.9      FBS        7   
4       18  114.3      FBS        7   

                                          source_url  
0  https://www.ncaa.com/stats/football/fbs/curren...  
1  https://www.ncaa.com/stats/football/fbs/curren...  
2  https://www.ncaa.com/stats/football/fbs/curren... 

In [9]:
import time
import requests
import pandas as pd
import numpy as np
from scipy.stats import f_oneway
from io import StringIO

# divisions we want to compare
DIVISIONS = ["fbs", "fcs", "d2", "d3"]

# selected stats by position
POSITION_STATS = {
    "QB": {
        "passing_yards": 453,
        "passing_td": 751,
        "yards_per_attempt": 908,
        "passing_efficiency": 8,
        "completion_percentage": 755,
        "passing_yards_per_game": 454
    },
    "RB": {
        "rushing_yards": 469,
        "rushing_td": 752,
        "rushing_yards_per_carry": 907,
        "rushing_yards_per_game": 7,
        "all_purpose_yards": 20,
        "scoring": 19
    },
    "WR": {
        "receiving_yards": 455,
        "receiving_td": 750,
        "receptions_per_game": 12,
        "yards_per_reception": 930,
        "receiving_yards_per_game": 7,
        "all_purpose_yards": 20
    },
    "DEF": {
        "tackles": 34,
        "tackles_for_loss": 39,
        "sacks": 36,
        "interceptions": 14,
        "passes_defended": 38,
        "forced_fumbles": 37
    }
}


def clean_number(x):
    """
    Convert table values into numbers when possible.
    Removes commas and percent signs.
    """
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = x.replace(",", "")
    x = x.replace("%", "")

    try:
        return float(x)
    except:
        return np.nan


def scrape_stat(stat_id, division):
    """
    Scrape one NCAA stat page for one division
    Return a DataFrame with player, value, and division columns
    """
    url = f"https://www.ncaa.com/stats/football/{division}/current/individual/{stat_id}"

    try:
        r = requests.get(url)

        tables = pd.read_html(StringIO(r.text))

        if len(tables) == 0:
            return None

        df = tables[0].copy()

        # make column names lowercase
        df.columns = [str(col).strip().lower() for col in df.columns]

        # find player column
        player_cols = [col for col in df.columns if "player" in col or "name" in col]
        if len(player_cols) == 0:
            return None

        player_col = player_cols[0]

        # use the last column as the main stat column
        stat_col = df.columns[-1]

        df = df[[player_col, stat_col]].copy()
        df.columns = ["player", "value"]

        # clean numeric values
        df["value"] = df["value"].apply(clean_number)

        # label division
        df["division"] = division.upper()

        return df

    except Exception:
        return None


def run_anova_on_stat(stat_name, stat_id):
    """
    Scrape one stat across all divisions and run one-way ANOVA.
    """
    frames = []

    for division in DIVISIONS:
        df = scrape_stat(stat_id, division)

        if df is not None:
            frames.append(df)

        time.sleep(0.05)

    # if nothing was scraped, return None
    if len(frames) == 0:
        return None

    # combine all divisions into one DataFrame
    all_df = pd.concat(frames, ignore_index=True)

    # collect numeric values by division
    groups = []
    means = {}

    for division in ["FBS", "FCS", "D2", "D3"]:
        vals = all_df[all_df["division"] == division]["value"].dropna()

        if len(vals) > 0:
            groups.append(vals)
            means[division] = vals.mean()
        else:
            means[division] = np.nan

    if len(groups) < 2:
        return None

    try:
        f_stat, p_value = f_oneway(*groups)

        result = {
            "stat": stat_name,
            "stat_id": stat_id,
            "F_stat": f_stat,
            "p_value": p_value,
            "mean_FBS": means["FBS"],
            "mean_FCS": means["FCS"],
            "mean_D2": means["D2"],
            "mean_D3": means["D3"]
        }

        return result

    except Exception:
        return None


def run_position_anova(position, stats_dict):
    """
    Run ANOVA for every chosen stat in one position group.
    """
    results = []

    for stat_name, stat_id in stats_dict.items():
        print("Testing", position, "-", stat_name)

        out = run_anova_on_stat(stat_name, stat_id)

        if out is not None:
            results.append(out)

    if len(results) == 0:
        return pd.DataFrame()

    results_df = pd.DataFrame(results)

    # sort by smallest p-value first
    results_df = results_df.sort_values("p_value")

    # add significance label
    results_df["significant"] = results_df["p_value"] < 0.05

    return results_df


# store final results for each position
ALL_RESULTS = {}
FINAL_SELECTION = {}

for position, stats_dict in POSITION_STATS.items():
    print("\n========================")
    print("RUNNING ANOVA FOR", position)
    print("========================")

    results_df = run_position_anova(position, stats_dict)

    ALL_RESULTS[position] = results_df

    if len(results_df) == 0:
        print("No results for", position)
        FINAL_SELECTION[position] = []
        continue

    # print full results for debugging
    print(results_df)
    print("\nSmaller summary:")
    print(results_df[["stat", "p_value", "significant"]])

    # keep only significant stats first
    sig_stats = results_df[results_df["significant"] == True]

    # if there are significant stats, use those
    if len(sig_stats) > 0:
        top_stats = list(sig_stats.head(4)["stat"])
    else:
        # otherwise, fall back to the 4 smallest p-values
        top_stats = list(results_df.head(4)["stat"])

    FINAL_SELECTION[position] = top_stats

print("\n========================")
print("FINAL SELECTED STATS")
print("========================")

for position, stats in FINAL_SELECTION.items():
    print(position + ":")
    
    for stat in stats:
        print("  -", stat)
    
    print()


RUNNING ANOVA FOR QB
Testing QB - passing_yards
Testing QB - passing_td
Testing QB - yards_per_attempt
Testing QB - passing_efficiency
Testing QB - completion_percentage
Testing QB - passing_yards_per_game
                     stat  stat_id     F_stat       p_value   mean_FBS  \
3      passing_efficiency        8  13.876738  3.049254e-08   152.9452   
2       yards_per_attempt      908  13.420481  6.291436e-08     8.3618   
5  passing_yards_per_game      454  12.812666  1.112308e-07   252.9338   
0           passing_yards      453  10.714364  1.491938e-06  3233.3800   
1              passing_td      751   9.884104  4.235130e-06    24.6400   
4   completion_percentage      755   8.645118  2.042574e-05     0.6750   

     mean_FCS      mean_D2    mean_D3  significant  
3   151.42400   156.146800   164.6326         True  
2     8.28080     9.015667     8.9230         True  
5   229.01500   249.759000   268.9972         True  
0  2772.94000  2801.680000  2905.4600         True  
1    22.5

In [71]:
import time
import requests
import pandas as pd
import numpy as np
from io import StringIO

# divisions to scrape
DIVISIONS = ["fbs", "fcs", "d2", "d3"]

# mapping to NCAA division codes used on stats.ncaa.org
DIVISION_CODES = {
    "fbs": "11.0",
    "fcs": "12.0",
    "d2": "2.0",
    "d3": "3.0"
}

# years to scrape
YEARS = list(range(2025, 1999, -1))

POSITION_STATS = {
    "QB": {
        "passing_yards": 453,
        "passing_td": 751,
        "yards_per_attempt": 908,
        "passing_efficiency": 8,
        "completion_percentage": 755,
        "passing_yards_per_game": 454
    },
    "RB": {
        "rushing_yards": 469,
        "rushing_td": 752,
        "rushing_yards_per_carry": 907,
        "rushing_yards_per_game": 7,
        "all_purpose_yards": 20,
        "scoring": 19
    },
    "WR": {
        "receiving_yards": 455,
        "receiving_td": 750,
        "receptions_per_game": 12,
        "yards_per_reception": 930,
        "receiving_yards_per_game": 7,
        "all_purpose_yards": 20
    },
    "DEF": {
        "tackles": 34,
        "tackles_for_loss": 39,
        "sacks": 36,
        "interceptions": 14,
        "passes_defended": 38,
        "forced_fumbles": 37
    }
}


def clean_number(x):

    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = x.replace(",", "")
    x = x.replace("%", "")

    try:
        return float(x)
    except:
        return np.nan


def scrape_stat_year(stat_id, division, year):

    division_code = DIVISION_CODES[division]

    url = (
        "https://stats.ncaa.org/rankings/national_ranking"
        f"?academic_year={year}.0"
        f"&division={division_code}"
        "&ranking_period=83.0"
        "&sport_code=MFB"
        f"&stat_seq={stat_id}.0"
    )

    try:

        r = requests.get(url, timeout=20)

        if r.status_code != 200:
            return None

        tables = pd.read_html(StringIO(r.text))

        if len(tables) == 0:
            return None

        df = tables[0].copy()

        df.columns = [str(col).strip().lower() for col in df.columns]

        player_cols = [c for c in df.columns if "player" in c or "name" in c]

        if len(player_cols) == 0:
            return None

        player_col = player_cols[0]

        stat_col = df.columns[-1]

        df = df[[player_col, stat_col]].copy()
        df.columns = ["player", "value"]

        df["value"] = df["value"].apply(clean_number)

        df["division"] = division.upper()
        df["year"] = year
        df["stat_id"] = stat_id
        df["source_url"] = url

        return df

    except Exception:
        return None


def scrape_role_stat(role, stat_name, stat_id):

    frames = []

    for division in DIVISIONS:
        for year in YEARS:

            print("Scraping", role, "-", stat_name, "-", division.upper(), "-", year)

            df = scrape_stat_year(stat_id, division, year)

            if df is not None:
                df["role"] = role
                df["stat"] = stat_name
                frames.append(df)

            time.sleep(0.05)

    if len(frames) == 0:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


def scrape_all_position_stats():

    all_frames = []

    for role, stats_dict in POSITION_STATS.items():

        for stat_name, stat_id in stats_dict.items():

            df = scrape_role_stat(role, stat_name, stat_id)

            if len(df) > 0:
                all_frames.append(df)

    if len(all_frames) == 0:
        return pd.DataFrame()

    return pd.concat(all_frames, ignore_index=True)


# run full scrape
ncaa_stats_df = scrape_all_position_stats()

print("\n========================")
print("SCRAPE COMPLETE")
print("========================")

print("\nTotal rows scraped:")
print(len(ncaa_stats_df))

print("\nColumns:")
print(ncaa_stats_df.columns.tolist())

print("\nPreview:")
print(ncaa_stats_df.head(20))


print("\n========================")
print("ROWS PER ROLE / STAT")
print("========================")

summary = (
    ncaa_stats_df
    .groupby(["role", "stat"])
    .size()
    .reset_index(name="rows")
)

print(summary)

Scraping QB - passing_yards - FBS - 2025
Scraping QB - passing_yards - FBS - 2024
Scraping QB - passing_yards - FBS - 2023
Scraping QB - passing_yards - FBS - 2022
Scraping QB - passing_yards - FBS - 2021
Scraping QB - passing_yards - FBS - 2020
Scraping QB - passing_yards - FBS - 2019
Scraping QB - passing_yards - FBS - 2018
Scraping QB - passing_yards - FBS - 2017
Scraping QB - passing_yards - FBS - 2016
Scraping QB - passing_yards - FBS - 2015
Scraping QB - passing_yards - FBS - 2014
Scraping QB - passing_yards - FBS - 2013
Scraping QB - passing_yards - FBS - 2012
Scraping QB - passing_yards - FBS - 2011
Scraping QB - passing_yards - FBS - 2010
Scraping QB - passing_yards - FBS - 2009
Scraping QB - passing_yards - FBS - 2008
Scraping QB - passing_yards - FBS - 2007
Scraping QB - passing_yards - FBS - 2006
Scraping QB - passing_yards - FBS - 2005
Scraping QB - passing_yards - FBS - 2004
Scraping QB - passing_yards - FBS - 2003
Scraping QB - passing_yards - FBS - 2002
Scraping QB - pa

KeyboardInterrupt: 

In [78]:
import time
import requests
import pandas as pd
import numpy as np
from io import StringIO
from concurrent.futures import ThreadPoolExecutor, as_completed

DIVISIONS = ["fbs","fcs","d2","d3"]

DIVISION_CODES = {
    "fbs":"11.0",
    "fcs":"12.0",
    "d2":"2.0",
    "d3":"3.0"
}

YEARS = list(range(2026,1999,-1))

POSITION_STATS = {
    "QB":{
        "passing_efficiency":8,
        "yards_per_attempt":908,
        "passing_yards_per_game":454,
        "passing_yards":453
    },
    "RB":{
        "all_purpose_yards":20,
        "rushing_td":752,
        "rushing_yards_per_carry":907,
        "rushing_yards_per_game":7
    },
    "WR":{
        "receptions_per_game":12,
        "all_purpose_yards":20,
        "yards_per_reception":930,
        "receiving_yards":455
    },
    "DEF":{
        "interceptions":14,
        "tackles_for_loss":39,
        "tackles":34,
        "sacks":36
    }
}


def clean_number(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip()
    x = x.replace(",", "")
    x = x.replace("%", "")

    try:
        return float(x)
    except Exception:
        return np.nan


def scrape_stat_year(stat_id, division, year):
    division_code = DIVISION_CODES[division]

    url = (
        "https://stats.ncaa.org/rankings/national_ranking"
        f"?academic_year={year}.0"
        f"&division={division_code}"
        "&ranking_period=83.0"
        "&sport_code=MFB"
        f"&stat_seq={stat_id}.0"
    )

    try:
        r = requests.get(url, timeout=10)
        r.raise_for_status()

        tables = pd.read_html(StringIO(r.text))

        if len(tables) == 0:
            return None

        table = None

        for t in tables:
            temp = t.copy()

            if isinstance(temp.columns, pd.MultiIndex):
                temp.columns = [
                    " ".join([str(x) for x in col if str(x) != "nan"]).strip().lower()
                    for col in temp.columns
                ]
            else:
                temp.columns = [str(c).strip().lower() for c in temp.columns]

            if temp.shape[1] >= 2:
                table = temp
                break

        if table is None:
            return None

        player_col = table.columns[1]
        stat_col = table.columns[-1]

        df = table[[player_col, stat_col]].copy()
        df.columns = ["player", "value"]

        df["value"] = df["value"].apply(clean_number)
        df["division"] = division.upper()
        df["year"] = year
        df["stat_id"] = stat_id
        df["source_url"] = url

        return df

    except Exception:
        return None


def scrape_role_stat(role, stat_name, stat_id, max_workers=12):
    frames = []
    jobs = []

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for division in DIVISIONS:
            for year in YEARS:
                jobs.append(
                    executor.submit(scrape_stat_year, stat_id, division, year)
                )

        for future in as_completed(jobs):
            df = future.result()
            if df is not None and len(df) > 0:
                df["role"] = role
                df["stat"] = stat_name
                frames.append(df)

    if len(frames) == 0:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


def scrape_all_position_stats(max_workers=12):
    all_frames = []

    for role, stats_dict in POSITION_STATS.items():
        for stat_name, stat_id in stats_dict.items():
            df = scrape_role_stat(role, stat_name, stat_id, max_workers=max_workers)

            if len(df) > 0:
                all_frames.append(df)

    if len(all_frames) == 0:
        return pd.DataFrame()

    return pd.concat(all_frames, ignore_index=True)

df = scrape_all_position_stats()
print("Rows collected:", len(df))

def quick_test_debug():
    stat_id = 8
    division = "fbs"
    year = 2025

    division_code = DIVISION_CODES[division]

    url = (
        "https://stats.ncaa.org/rankings/national_ranking"
        f"?academic_year={year}.0"
        f"&division={division_code}"
        "&ranking_period=83.0"
        "&sport_code=MFB"
        f"&stat_seq={stat_id}.0"
    )

    print("Testing URL:")
    print(url)

    try:
        r = requests.get(url, timeout=10)
        print("Status code:", r.status_code)
        print("Final URL:", r.url)
        print("First 500 chars of response:")
        print(r.text[:500])

        tables = pd.read_html(StringIO(r.text))
        print("Number of tables found:", len(tables))

        for i, t in enumerate(tables):
            print(f"\n--- Table {i} ---")
            print("Shape:", t.shape)
            print(t.head())

    except Exception as e:
        print("ERROR:", repr(e))

quick_test_debug()

Rows collected: 0
Testing URL:
https://stats.ncaa.org/rankings/national_ranking?academic_year=2025.0&division=11.0&ranking_period=83.0&sport_code=MFB&stat_seq=8.0
Status code: 403
Final URL: https://stats.ncaa.org/rankings/national_ranking?academic_year=2025.0&division=11.0&ranking_period=83.0&sport_code=MFB&stat_seq=8.0
First 500 chars of response:
<HTML><HEAD>
<TITLE>Access Denied</TITLE>
</HEAD><BODY>
<H1>Access Denied</H1>
 
You don't have permission to access "http&#58;&#47;&#47;stats&#46;ncaa&#46;org&#47;rankings&#47;national&#95;ranking&#63;" on this server.<P>
Reference&#32;&#35;18&#46;465ed617&#46;1773356414&#46;22a1f904
<P>https&#58;&#47;&#47;errors&#46;edgesuite&#46;net&#47;18&#46;465ed617&#46;1773356414&#46;22a1f904</P>
</BODY>
</HTML>

ERROR: ValueError('No tables found')


In [80]:
def scrape_stat_year(stat_id, division, year):
    """
    Scrape one stat for one division and one year
    using the NCAA API wrapper.
    """

    api_url = f"https://ncaa-api.henrygd.me/stats/football/{division}/{year}/individual/{stat_id}"

    try:
        r = requests.get(api_url, timeout=8)

        if r.status_code != 200:
            return None

        data = r.json()

        # figure out where the rows are stored
        if isinstance(data, dict) and "data" in data:
            rows = data["data"]
        elif isinstance(data, dict) and "rows" in data:
            rows = data["rows"]
        elif isinstance(data, list):
            rows = data
        else:
            return None

        if len(rows) == 0:
            return None

        df = pd.DataFrame(rows)

        # clean column names
        df.columns = [str(c).strip().lower() for c in df.columns]

        # find player column
        player_cols = [c for c in df.columns if "player" in c or "name" in c]

        if len(player_cols) == 0:
            return None

        player_col = player_cols[0]
        stat_col = df.columns[-1]

        df = df[[player_col, stat_col]]
        df.columns = ["player", "value"]

        # clean values
        df["value"] = df["value"].apply(clean_number)

        # metadata
        df["division"] = division.upper()
        df["year"] = year
        df["stat_id"] = stat_id
        df["source_url"] = api_url

        return df

    except requests.exceptions.Timeout:
        print("Timed out:", api_url)
        return None

    except requests.exceptions.RequestException:
        print("Request failed:", api_url)
        return None

    except Exception:
        return None

In [92]:
DIVISIONS=["fbs"]
YEARS=[2017]

small_test=scrape_role_stat("QB","passing_yards",453)

print(small_test.head())
print("Rows:",len(small_test))

Empty DataFrame
Columns: []
Index: []
Rows: 0


In [82]:
df = scrape_all_position_stats(max_workers=12)